# Progressive disclosure: controlling what the LLM sees

As an agent class grows, the LLM needs to know which methods and fields it can actually use. NOOA renders that view automatically via `doc(self)` and drops it into every prompt. This notebook shows the levers you can pull to hide the parts the LLM should not see: `@hidden`, leading underscores, `Annotated[T, hidden]`, `@spec(hidden=False)`, and the module-level `with hidden:` block.

## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. Replace `"your-api-key"` with a real key for hosted providers; local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
# model = get_llm_client("openai/openai/openai/gpt-5.5", api_key="your-api-key", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)


## A small agent

A `LibraryAgent` with a couple of tools and a private helper. The public methods look up books and check them out; the underscore-prefixed method is an internal helper.

In [2]:
import nooa
from nooa import Agent
from nooa.agentdoc import doc

class LibraryAgent(Agent, llm=model):
    """You help patrons find books in a small branch library."""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
        return self._lookup(query)

    def check_out(self, title: str, patron_id: str) -> str:
        """Record a checkout for the given patron and return a confirmation string."""
        return f"Checked out {title!r} to {patron_id}."

    async def help_patron(self, request: str) -> str:
        """Handle a patron request end to end: find the book, then check it out if they want it."""
        ...

    def _lookup(self, query: str) -> str:
        # Internal catalog lookup. Underscore-prefixed, so hidden by default.
        return f"Aisle 3, shelf B ({query})"

## What the LLM sees

`doc(self)` is the exact view the LLM is given for the agent. Notice that `_lookup` is absent — the leading underscore hides it by default.

In [3]:
agent = LibraryAgent()
print(doc(agent))

OTel tracing enabled: journal:http://localhost:5001
class LibraryAgent:
    """You help patrons find books in a small branch library."""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
    def check_out(self, title: str, patron_id: str) -> str:
        """Record a checkout for the given patron and return a confirmation string."""
    async def help_patron(self, request: str) -> str:
        """Handle a patron request end to end: find the book, then check it out if they want it."""


In [6]:
agent = LibraryAgent()
print(doc(agent))

class LibraryAgent:
    """You help patrons find books in a small branch library."""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
    async def help_patron(self, request: str) -> str:
        """Handle a patron request end to end: find the book, then check it out if they want it."""


## The full prompt

`print_prompt` renders the entire prompt that would be sent to the LLM for a specific method call. Look for the `<self>` block — that's the class API surface. Everything the model can call goes through this view.

In [4]:
await nooa.print_prompt(agent.help_patron, request="I'd like to borrow a book about beekeeping.")

=== SYSTEM PROMPT  [LibraryAgent] ===

<system_prompt expr="self._resolve_system_prompt()">
You help patrons find books in a small branch library.
</system_prompt>

<strategy_prompt>
## Strategy

Jupyter-like Python session. Parameters pre-loaded as locals; state persists across cells. Use `await` directly, `print`/`pprint` to debug, `doc(obj)` to inspect types. You MUST call a tool each turn — **plain-text responses do NOT end the session**. To finish, call `return_result(value)`. Repeated text-only responses will abort the run with an error.

**Your two tools:**
- `execute_python(code)` — run a code cell
- `return_result(value)` — submit your final answer (also callable from inside `execute_python`)

## When to use which tool

Use `return_result(...)` directly for simple answers determinable from the inputs alone (yes/no, one field, a single lookup).

Use `execute_python(...)` for lists/batches, arithmetic, multi-step computation, transforms, or iteration. Always iterate in code — ne

## Hiding a public method with `@hidden`

Suppose `check_out` should only be invoked by the harness, not chosen by the LLM. Decorate it with `@hidden` and redefine the class. The method still exists and still runs when you call it directly — but it disappears from the `<self>` block.

In [5]:
from nooa import hidden

class LibraryAgent(Agent, llm=model):
    """You help patrons find books in a small branch library."""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
        return self._lookup(query)

    @hidden
    def check_out(self, title: str, patron_id: str) -> str:
        """Record a checkout for the given patron and return a confirmation string."""
        return f"Checked out {title!r} to {patron_id}."

    async def help_patron(self, request: str) -> str:
        """Handle a patron request end to end: find the book, then check it out if they want it."""
        ...

    def _lookup(self, query: str) -> str:
        return f"Aisle 3, shelf B ({query})"

agent = LibraryAgent()
await nooa.print_prompt(agent.help_patron, request="I'd like to borrow a book about beekeeping.")

=== SYSTEM PROMPT  [LibraryAgent] ===

<system_prompt expr="self._resolve_system_prompt()">
You help patrons find books in a small branch library.
</system_prompt>

<strategy_prompt>
## Strategy

Jupyter-like Python session. Parameters pre-loaded as locals; state persists across cells. Use `await` directly, `print`/`pprint` to debug, `doc(obj)` to inspect types. You MUST call a tool each turn — **plain-text responses do NOT end the session**. To finish, call `return_result(value)`. Repeated text-only responses will abort the run with an error.

**Your two tools:**
- `execute_python(code)` — run a code cell
- `return_result(value)` — submit your final answer (also callable from inside `execute_python`)

## When to use which tool

Use `return_result(...)` directly for simple answers determinable from the inputs alone (yes/no, one field, a single lookup).

Use `execute_python(...)` for lists/batches, arithmetic, multi-step computation, transforms, or iteration. Always iterate in code — ne

`check_out` is no longer in the `<self>` block. The LLM will not know it exists, but your Python code can still call `agent.check_out(...)` directly.

## The other levers

Five ways to control visibility. All follow the same rule: **visible by default, hide explicitly** — with one exception, leading underscores, which follow Python's own convention.

| Lever | Where it applies | Effect |
| --- | --- | --- |
| `_leading_underscore` | methods, fields | hidden by default |
| `@hidden` | methods | hide a public method |
| `Annotated[T, hidden]` | fields | hide a field |
| `@spec(hidden=False)` | private methods | expose a `_name` method |
| `with hidden:` | module-level imports | hide imports from the LLM's execution context |

A single class showing four of them together:

In [ ]:
from typing import Annotated
from nooa import Agent, hidden, spec

class LibraryAgent(Agent, llm=model):
    """You help patrons find books in a small branch library."""

    # Visible field — the LLM sees it in <self>.
    branch: str = "Main St."

    # Hidden field — secrets, connection handles, anything the LLM should not read.
    api_token: Annotated[str, hidden] = ""

    def find_book(self, query: str) -> str:
        """Return the shelf location for the book matching the query."""
        return self._lookup(query)

    @hidden
    def check_out(self, title: str, patron_id: str) -> str:
        """Runs on the harness side only."""
        return f"Checked out {title!r} to {patron_id}."

    @spec(hidden=False)
    def _catalog_stats(self) -> dict:
        """Return catalog stats. Underscore-prefixed but explicitly exposed."""
        return {"total": 12_000, "available": 8_412}

    def _lookup(self, query: str) -> str:
        # Still hidden — no override.
        return f"Aisle 3, shelf B ({query})"

print(doc(LibraryAgent))

In the rendered docs: `branch` is visible, `api_token` is gone, `check_out` is gone, `_catalog_stats` is visible despite its underscore, and `_lookup` stays hidden.

## Hiding module-level imports

The prompt's `<execution_context>` block lists everything imported in the agent's module — those names are available to the LLM when it writes code. Wrap sensitive or noisy imports in `with hidden:` to keep them out of that list. This is a module-level construct, so it belongs at the top of a `.py` file:

```python
# my_agent.py
from nooa import Agent, hidden

with hidden:
    import boto3
    from mycompany.secrets import load_credentials

class LibraryAgent(Agent):
    ...
```

`boto3` and `load_credentials` are still usable from your methods, but they won't appear in the LLM's execution context.

> **Takeaway:** visible by default, hide explicitly — the same rule Python uses for public APIs, extended to your agent's LLM view.